# SMARD.de Energie-Analyse — Deutschland

Dieses Notebook analysiert deutsche Strommarktdaten über die öffentliche **SMARD API**
des Bundesnetzagentur-Portals [smard.de](https://www.smard.de).

**Analysen (A1–A4) spiegeln die ENTSO-E Analysen auf SMARD-Basis:**

| # | Analyse |
|---|--------|
| A1 | Preis-Einspeise-Kurve je Brennstofftyp |
| A2 | Reaktionszeit auf Preissignal (±6 h) |
| A3 | Dauer vs. Intensität — Negativpreis-Episoden |
| A4 | Merit-Order-Fingerabdruck |

> **Hinweis zu Block-Ebene (Analyse 1):** Die SMARD API liefert Aggregate je Brennstofftyp.
> Für echte Block-Ebene stellt SMARD CSV-Downloads bereit (Downloadcenter → Marktdaten).
> Diese können mit `pd.read_csv()` geladen und direkt in die Analysen eingesetzt werden.

---
## Setup & SMARD-API-Client

Die SMARD API ist eine **öffentliche REST-API** — kein API-Key nötig.

**Endpunktstruktur:**
```
Index (verfügbare Zeitstempel):
  GET /app/chart_data/{indicator}/DE/index_{resolution}.json

Daten für einen Zeitblock:
  GET /app/chart_data/{indicator}/DE/{indicator}_DE_{resolution}_{timestamp}.json
```
Daten sind in **Wochenblöcken** (Unix-ms-Timestamps) organisiert.
Die Hilfsfunktion `smard_fetch()` holt automatisch alle benötigten Blöcke.

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

SMARD_BASE = 'https://www.smard.de/app/chart_data'

# ── Indikator-IDs ─────────────────────────────────────────────────────
INDIKATOREN = {
    'Braunkohle':            4066,
    'Kernenergie':           4067,
    'Erdgas':                4068,
    'Pumpspeicher':          4069,
    'Wind Offshore':         4070,
    'Wind Onshore':          4071,
    'Solar':                 4072,
    'Sonstige Erneuerbare':  4073,
    'Wasserkraft':           4074,
    'Biomasse':              4075,
    'Steinkohle':            4076,
    'Sonstige Konventionelle': 4077,
    'Day-Ahead Preis':       8009,
}

# ── Farben ────────────────────────────────────────────────────────────
SMARD_FARBEN = {
    'Braunkohle':              '#8B4513',
    'Kernenergie':             '#9400D3',
    'Erdgas':                  '#FF8C00',
    'Pumpspeicher':            '#008B8B',
    'Wind Offshore':           '#1E90FF',
    'Wind Onshore':            '#87CEEB',
    'Solar':                   '#FFD700',
    'Sonstige Erneuerbare':    '#32CD32',
    'Wasserkraft':             '#00CED1',
    'Biomasse':                '#228B22',
    'Steinkohle':              '#696969',
    'Sonstige Konventionelle': '#A9A9A9',
    'Day-Ahead Preis':         '#E8A838',
}

# ── API-Client ────────────────────────────────────────────────────────
def smard_index(indicator_id: int, resolution: str = 'hour') -> list:
    """Gibt Liste verfügbarer Zeitstempel (Unix ms) für einen Indikator zurück."""
    url = f'{SMARD_BASE}/{indicator_id}/DE/index_{resolution}.json'
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    return r.json().get('timestamps', [])


def smard_block(indicator_id: int, timestamp_ms: int,
                resolution: str = 'hour') -> pd.Series:
    """Holt einen Datenblock (eine Woche) als Pandas Series."""
    url = (f'{SMARD_BASE}/{indicator_id}/DE/'
           f'{indicator_id}_DE_{resolution}_{timestamp_ms}.json')
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    series_data = r.json().get('series', [])
    if not series_data:
        return pd.Series(dtype=float)
    ts  = pd.to_datetime([p[0] for p in series_data], unit='ms', utc=True)
    val = [p[1] for p in series_data]
    return pd.Series(val, index=ts, dtype=float)


def smard_fetch(indicator_id: int, start: pd.Timestamp, end: pd.Timestamp,
                resolution: str = 'hour') -> pd.Series:
    """Holt alle Datenblöcke für einen Zeitraum und konkateniert sie."""
    start_ms = int(start.timestamp() * 1000)
    end_ms   = int(end.timestamp()   * 1000)

    timestamps = smard_index(indicator_id, resolution)
    # Nur Blöcke, deren Startzeit ≤ end und die potentiell Daten in [start, end] haben
    # Jeder Block umfasst ~1 Woche (604_800_000 ms)
    BLOCK_MS = 604_800_000
    relevant = [t for t in timestamps
                if t <= end_ms and t + BLOCK_MS >= start_ms]

    parts = []
    for t in relevant:
        try:
            blk = smard_block(indicator_id, t, resolution)
            parts.append(blk)
        except Exception:
            pass

    if not parts:
        return pd.Series(dtype=float)

    full = pd.concat(parts).sort_index()
    full = full[~full.index.duplicated(keep='first')]
    # Auf Berlin-Zeitzone konvertieren und auf Zeitraum schneiden
    full.index = full.index.tz_convert('Europe/Berlin')
    return full.loc[start:end]


print('SMARD API Client initialisiert.')
print(f'Verfügbare Indikatoren: {list(INDIKATOREN.keys())}')

---
## Daten laden: Preise und Erzeugung je Brennstofftyp

Alle Zeitreihen werden für denselben Zeitraum geladen wie im ENTSO-E Notebook
(oder nach Bedarf angepasst).

In [ ]:
# ── Zeitraum (anpassen nach Bedarf) ─────────────────────────────────────
START = pd.Timestamp('2024-06-01', tz='Europe/Berlin')
END   = pd.Timestamp('2024-07-01', tz='Europe/Berlin')
print(f'Zeitraum: {START.date()} bis {END.date()}')

# ── Day-Ahead Preis ───────────────────────────────────────────────────
print('Lade Day-Ahead Preise...')
smard_preis = smard_fetch(INDIKATOREN['Day-Ahead Preis'], START, END)
print(f'  Preise: {len(smard_preis)} Datenpunkte')
print(f'  Min: {smard_preis.min():.2f} €/MWh | Max: {smard_preis.max():.2f} €/MWh')
print(f'  Negativpreise: {(smard_preis < 0).sum()} Stunden')

# ── Erzeugung je Brennstofftyp ───────────────────────────────────────
print('\nLade Erzeugungsdaten je Brennstofftyp...')
smard_gen = {}
for name, iid in INDIKATOREN.items():
    if name == 'Day-Ahead Preis':
        continue
    try:
        s = smard_fetch(iid, START, END)
        if not s.empty:
            smard_gen[name] = s
            print(f'  {name:<25}: {len(s):>5} Datenpunkte  Ø {s.mean():>8,.0f} MW')
    except Exception as e:
        print(f'  {name}: Fehler — {e}')

# ── Gemeinsamen DataFrame aufbauen ───────────────────────────────────
smard_df = pd.DataFrame(smard_gen)
smard_df = smard_df.resample('1h').mean()   # auf Stundenbasis
smard_df = smard_df.fillna(0)

# Preise ebenfalls auf Stundenbasis
smard_preis_h = smard_preis.resample('1h').mean()

print(f'\nGemeinsamer DataFrame: {smard_df.shape[0]} Stunden × {smard_df.shape[1]} Typen')
smard_df.head(3)

In [ ]:
# ── Übersicht: Stacked-Area-Chart ──────────────────────────────────────
erneuerbare = ['Wind Onshore','Wind Offshore','Solar','Wasserkraft',
               'Biomasse','Sonstige Erneuerbare']
konventionell = ['Braunkohle','Steinkohle','Erdgas','Kernenergie',
                 'Pumpspeicher','Sonstige Konventionelle']

cols_plot = [c for c in erneuerbare + konventionell if c in smard_df.columns]
farben_plot = [SMARD_FARBEN.get(c, '#888888') for c in cols_plot]

fig, ax = plt.subplots(figsize=(16, 6))
ax.stackplot(smard_df.index,
             [smard_df[c] for c in cols_plot],
             labels=cols_plot, colors=farben_plot, alpha=0.85)
ax2p = ax.twinx()
ax2p.plot(smard_preis_h.index, smard_preis_h.values,
          color='black', linewidth=1, alpha=0.6, label='Preis €/MWh')
ax2p.axhline(0, color='red', linestyle='--', linewidth=0.7)
ax2p.set_ylabel('€/MWh')
ax.set_title('Stromerzeugung Deutschland nach Energieträger (SMARD)', fontsize=13, fontweight='bold')
ax.set_ylabel('MW')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
ax.legend(loc='upper left', bbox_to_anchor=(1.06, 1), fontsize=9)
ax2p.legend(loc='upper left', bbox_to_anchor=(1.06, 0.3), fontsize=9)
plt.tight_layout()
plt.savefig('smard_ueberblick.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Analyse 1 (SMARD): Preis-Einspeise-Kurve je Brennstofftyp

**Scatterplot MW vs. Day-Ahead-Preis** — zeigt implizite Abschaltschwellen je Typ.

> **Block-Ebene:** Für die Einzelblock-Ansicht können SMARD-CSV-Downloads
> (Downloadcenter → *Erzeugung je Einspeiseeinheit*) als DataFrame geladen werden:
> ```python
> df_blocks = pd.read_csv('smard_blocks.csv', sep=';', decimal=',',
>                          parse_dates=['Datum von'], index_col='Datum von')
> ```
> Die restliche Analyse funktioniert identisch mit dem Aggregat.

In [ ]:
from matplotlib.patches import Patch

common_a1 = smard_df.index.intersection(smard_preis_h.index)
gen_a1    = smard_df.loc[common_a1]
pr_a1     = smard_preis_h.loc[common_a1]

konv_s = [c for c in konventionell if c in gen_a1.columns and gen_a1[c].mean() > 1]

def scatter_mw_preis(ax, preis_arr, mw_arr, name, farbe):
    mask_neg = preis_arr < 0
    mask_pos = preis_arr >= 0
    if mask_pos.sum() > 0:
        ax.scatter(preis_arr[mask_pos], mw_arr[mask_pos],
                   alpha=0.25, s=8, color=farbe, rasterized=True)
    if mask_neg.sum() > 0:
        ax.scatter(preis_arr[mask_neg], mw_arr[mask_neg],
                   alpha=0.7, s=14, color='#E74C3C', rasterized=True)
    ax.axvline(0, color='black', linestyle='--', linewidth=0.8)
    # Abschaltschwelle
    nenn = mw_arr.max()
    aktiv = mw_arr > nenn * 0.05
    if aktiv.sum() > 5:
        schwelle = preis_arr[aktiv].min()
        ax.axvline(schwelle, color='#F39C12', linestyle=':', linewidth=1.3,
                   label=f'Schwelle: {schwelle:.0f} €')
    ax.set_title(name, fontsize=9, fontweight='bold')
    ax.set_xlabel('€/MWh', fontsize=8)
    ax.set_ylabel('MW', fontsize=8)
    ax.legend(fontsize=7)

if konv_s:
    ncols = min(3, len(konv_s))
    nrows = (len(konv_s) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 5))
    axes = np.array(axes).flatten() if len(konv_s) > 1 else [axes]

    for i, typ in enumerate(konv_s):
        scatter_mw_preis(axes[i], pr_a1.values, gen_a1[typ].values,
                          typ, SMARD_FARBEN.get(typ, '#4A90D9'))

    for j in range(len(konv_s), len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(
        'A1 (SMARD) — Preis-Einspeise-Kurve je Brennstofftyp\n'
        'Typ-Farbe: Preis ≥ 0  |  Rot: Negativpreis  |  Orange: implizite Abschaltschwelle',
        fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('smard_a1_preiskurve.png', dpi=120, bbox_inches='tight')
    plt.show()

    # Abschaltschwellen-Tabelle
    print('\n── Implizite Abschaltschwellen (tiefster Preis bei Einspeisung > 5 % Nennlast) ──')
    for typ in konv_s:
        mw  = gen_a1[typ].values
        pr  = pr_a1.values
        aktiv = mw > mw.max() * 0.05
        if aktiv.sum() > 0:
            print(f'  {typ:<28}: {pr[aktiv].min():>8.2f} €/MWh  '
                  f'(Nennlast-Näherung: {mw.max():,.0f} MW)')
else:
    print('Keine konventionellen Daten verfügbar.')

---
## Analyse 2 (SMARD): Reaktionszeit auf Preissignal

Stundengenaue Einspeisekurve ±6 h um jeden Eintritt in eine Negativpreisphase.  
Zeigt, ob konventionelle Typen *vor* dem Negativpreis drosseln oder erst danach.

In [ ]:
# ── Eintrittszeiten ─────────────────────────────────────────────────────
neg_bin_s   = (smard_preis_h < 0).astype(int)
eintritt_s  = neg_bin_s.diff() == 1
entry_s     = smard_preis_h.index[eintritt_s]
print(f'Negativpreis-Eintritte (SMARD): {len(entry_s)}')

KONV_S = [c for c in konventionell if c in smard_df.columns and smard_df[c].mean() > 1]
FENSTER = 6

if not entry_s.empty and KONV_S:
    fig, axes = plt.subplots(len(entry_s), 1,
                              figsize=(14, 5 * len(entry_s)), squeeze=False)
    axes = axes.flatten()

    for idx, t_e in enumerate(entry_s):
        ax  = axes[idx]
        t0  = t_e - pd.Timedelta(hours=FENSTER)
        t1  = t_e + pd.Timedelta(hours=FENSTER)

        f_pr  = smard_preis_h.loc[t0:t1]
        f_gen = smard_df.loc[t0:t1]

        rel_pr  = [(t - t_e).total_seconds() / 3600 for t in f_pr.index]
        rel_gen = [(t - t_e).total_seconds() / 3600 for t in f_gen.index]

        ax2 = ax.twinx()
        ax2.plot(rel_pr, f_pr.values, color='black', lw=1.5, ls='--', alpha=0.65)
        ax2.axhline(0, color='red', lw=0.7, alpha=0.5)
        ax2.fill_between(rel_pr, f_pr.values, 0,
                         where=[v < 0 for v in f_pr.values], alpha=0.12, color='red')
        ax2.set_ylabel('€/MWh')

        for typ in KONV_S:
            werte = f_gen[typ].values
            wmax  = werte.max()
            if wmax > 0:
                ax.plot(rel_gen, werte / wmax * 100,
                        label=typ, color=SMARD_FARBEN.get(typ, '#888'), lw=1.8)

        ax.axvline(0, color='red', lw=1.5, alpha=0.8)
        ax.set_xlim(-FENSTER, FENSTER)
        ax.set_xlabel('Stunden relativ zum Negativpreis-Eintritt')
        ax.set_ylabel('Einspeisung (% Fenstermax.)')
        ax.set_title(f'Eintritt: {t_e.strftime("%d.%m.%Y %H:%M")}  '
                     f'Preis: {smard_preis_h.loc[t_e]:.1f} €/MWh', fontsize=10)
        ax.legend(loc='upper left', fontsize=7)

    plt.suptitle('A2 (SMARD) — Reaktionszeit auf Negativpreis-Signal\n'
                 'Rote Linie = Negativpreis-Eintritt | Werte normiert auf Fenstermaximum',
                 fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('smard_a2_reaktionszeit.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('Keine Negativpreis-Eintritte oder keine konventionellen Daten.')

---
## Analyse 3 (SMARD): Dauer vs. Intensität — Negativpreis-Episoden

Für jede zusammenhängende Negativpreisphase: Aktivitätsgrad je Brennstofftyp,
Episodendauer und -tiefe. Zeigt, wer am längsten durchhält.

In [ ]:
def extrahiere_episoden_s(preisserie):
    eps, in_ep, t_start = [], False, None
    for t, v in preisserie.items():
        if v < 0 and not in_ep:
            t_start, in_ep = t, True
        elif v >= 0 and in_ep:
            seg = preisserie.loc[t_start:t].iloc[:-1]
            eps.append({'start': t_start, 'end': t,
                        'dauer_h': (t - t_start).total_seconds() / 3600,
                        'mean_preis': seg.mean(), 'min_preis': seg.min()})
            in_ep = False
    if in_ep:
        seg = preisserie.loc[t_start:]
        eps.append({'start': t_start, 'end': preisserie.index[-1],
                    'dauer_h': (preisserie.index[-1] - t_start).total_seconds() / 3600,
                    'mean_preis': seg.mean(), 'min_preis': seg.min()})
    return pd.DataFrame(eps)

eps_s = extrahiere_episoden_s(smard_preis_h)
print(f'Episoden: {len(eps_s)}')
if not eps_s.empty:
    print(eps_s[['start','end','dauer_h','mean_preis','min_preis']].to_string())

# ── Aktivitätsgrad je Episode ────────────────────────────────────────
if not eps_s.empty and KONV_S:
    kap_ref_s = smard_df[KONV_S].quantile(0.95)

    for typ in KONV_S:
        akt = []
        for _, ep in eps_s.iterrows():
            seg = smard_df.loc[ep['start']:ep['end'], typ]
            ref = kap_ref_s[typ] if kap_ref_s[typ] > 0 else 1
            akt.append(seg.mean() / ref * 100)
        eps_s[f'aktiv_{typ}'] = akt

    # ── Scatter: Dauer vs. Mittelpreis, Farbe = Aktivitätsgrad ──────
    ncols = min(3, len(KONV_S))
    nrows = (len(KONV_S) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    axes = np.array(axes).flatten() if len(KONV_S) > 1 else [axes]

    for ax, typ in zip(axes, KONV_S):
        col   = f'aktiv_{typ}'
        farbe = SMARD_FARBEN.get(typ, '#888')
        sc = ax.scatter(eps_s['dauer_h'], eps_s['mean_preis'],
                        c=eps_s[col], cmap='RdYlGn_r',
                        s=eps_s['dauer_h'] * 15 + 30,
                        edgecolors=farbe, linewidths=1.2, alpha=0.85,
                        vmin=0, vmax=100)
        plt.colorbar(sc, ax=ax, label='Aktivität (%)')
        ax.axhline(0, color='red', ls='--', lw=0.8)
        for _, row in eps_s.iterrows():
            ax.annotate(row['start'].strftime('%d.%m'),
                        (row['dauer_h'], row['mean_preis']), fontsize=6, ha='center')
        ax.set_xlabel('Episodendauer (h)')
        ax.set_ylabel('Ø Preis in Episode (€/MWh)')
        ax.set_title(typ, fontsize=10, fontweight='bold')

    for j in range(len(KONV_S), len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('A3 (SMARD) — Dauer vs. Intensität je Negativpreis-Episode\n'
                 'Punktgröße = Dauer | Farbe = Aktivitätsgrad',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('smard_a3_episoden.png', dpi=120, bbox_inches='tight')
    plt.show()

    # Durchhalte-Ranking
    dh_s = {typ: eps_s[f'aktiv_{typ}'].mean() for typ in KONV_S}
    dh_ser = pd.Series(dh_s).sort_values()
    fig, ax = plt.subplots(figsize=(9, 4))
    dh_ser.plot(kind='barh', ax=ax,
                color=[SMARD_FARBEN.get(t, '#888') for t in dh_ser.index],
                edgecolor='white')
    ax.set_xlabel('Ø Aktivitätsgrad bei Negativpreisen (%)')
    ax.axvline(50, color='gray', ls=':', lw=1)
    ax.set_title('A3 (SMARD) — Durchhaltezeit-Ranking', fontweight='bold')
    plt.tight_layout()
    plt.savefig('smard_a3_ranking.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('Keine Episoden oder keine konventionellen Daten.')

---
## Analyse 4 (SMARD): Merit-Order-Fingerabdruck

Aggregierte Einspeisung aller Typen gegen Preis — gestapelte Kurve zeigt implizit
die Merit-Order. Bei welchem Preis fällt Braunkohle raus? Bei welchem Gas?

**Methode:** Preisbins (25 €/MWh-Schritte) → mittlere Einspeisung je Typ.

In [ ]:
# ── Preisbins ───────────────────────────────────────────────────────────
common_a4 = smard_df.index.intersection(smard_preis_h.index)
gen_a4    = smard_df.loc[common_a4].copy()
pr_a4     = smard_preis_h.loc[common_a4]

p_min_s = int(pr_a4.min() // 25) * 25
p_max_s = int(pr_a4.max() // 25 + 1) * 25
bins_s  = list(range(p_min_s, p_max_s + 25, 25))
labs_s  = [str(b) for b in bins_s[:-1]]

gen_a4['_bin'] = pd.cut(pr_a4, bins=bins_s, labels=labs_s, right=False).values
merit_s = gen_a4.groupby('_bin', observed=True).mean()
merit_s.index = merit_s.index.astype(str)

alle_typen_s = erneuerbare + konventionell
plot_cols_s  = [c for c in alle_typen_s if c in merit_s.columns]
farben_s     = [SMARD_FARBEN.get(c, '#888') for c in plot_cols_s]

# ── Gestapeltes Balkendiagramm ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 6))
bottom_s = np.zeros(len(merit_s))
x_s      = np.arange(len(merit_s))

for col, farbe in zip(plot_cols_s, farben_s):
    vals = merit_s[col].fillna(0).values
    ax.bar(x_s, vals, bottom=bottom_s, label=col, color=farbe,
           edgecolor='white', linewidth=0.3)
    bottom_s += vals

ax.set_xticks(x_s)
ax.set_xticklabels(merit_s.index, rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Preisbin (€/MWh, untere Grenze)')
ax.set_ylabel('Ø Einspeisung (MW)')
ax.set_title('A4 (SMARD) — Merit-Order-Fingerabdruck: Einspeisung je Preisbin\n'
             'Zeigt bei welchem Preis ein Typ aus der Merit-Order fällt',
             fontweight='bold')
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=9)
plt.tight_layout()
plt.savefig('smard_a4_merit_order.png', dpi=120, bbox_inches='tight')
plt.show()

# ── Linienplot: Konventionelle Typen ────────────────────────────────
konv_plot_s = [c for c in konventionell if c in merit_s.columns
               and merit_s[c].max() > 100]

if konv_plot_s:
    fig, ax = plt.subplots(figsize=(14, 5))
    for typ in konv_plot_s:
        ax.plot(merit_s.index, merit_s[typ].fillna(0),
                marker='o', ms=5, lw=2, label=typ,
                color=SMARD_FARBEN.get(typ, '#888'))
    ax.axvline(str(0) if str(0) in merit_s.index else labs_s[0],
               color='red', ls='--', lw=1, alpha=0.7, label='0 €/MWh')
    ax.set_xlabel('Preisbin (€/MWh)')
    ax.set_ylabel('Ø Einspeisung (MW)')
    ax.set_title('A4 (SMARD) — Konventionelle Merit-Order-Kurven', fontweight='bold')
    ax.legend(fontsize=9)
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.savefig('smard_a4_merit_konventionell.png', dpi=120, bbox_inches='tight')
    plt.show()

    # Quantitativer Vergleich: mittlere Einspeisung bei pos. vs. neg. Preisen
    print('\n── Einspeisung konventioneller Typen: Positiv- vs. Negativpreise ──')
    for typ in konv_plot_s:
        m_pos2 = gen_a4.loc[pr_a4 >= 0, typ].mean()
        m_neg2 = gen_a4.loc[pr_a4 <  0, typ].mean()
        if m_pos2 > 0:
            print(f'  {typ:<28}: pos. Ø {m_pos2:>7,.0f} MW  |  '
                  f'neg. Ø {m_neg2:>7,.0f} MW  (Rückgang {(1-m_neg2/m_pos2)*100:.0f} %)')
else:
    print('Keine ausreichenden konventionellen Daten für Linienplot.')